In [2]:
import arcpy
from arcpy import env
import os
import numpy as np
from arcgis import GIS
from arcgis.features import GeoAccessor
from arcgis.features import GeoSeriesAccessor
import pandas as pd
# import geopandas as gpd

arcpy.env.overwriteOutput = True
arcpy.env.parallelProcessingFactor = "90%"

# show all columns
pd.options.display.max_columns = None

# pd.pivot_table(df, values='a', index='b', columns='c', aggfunc='sum', fill_value=0)
# pd.DataFrame.spatial.from_featureclass(???)  
# df.spatial.to_featureclass(location=???,sanitize_columns=False)  

# gsa = arcgis.features.GeoSeriesAccessor(df['SHAPE'])  
# df['AREA'] = gsa.area  # KNOW YOUR UNITS

In [3]:
# fill NA values in Spatially enabled dataframes (ignores SHAPE column)
def fill_na_sedf(df_with_shape_column, fill_value=0):
    if 'SHAPE' in list(df_with_shape_column.columns):
        df = df_with_shape_column.copy()
        shape_column = df['SHAPE'].copy()
        del df['SHAPE']
        return df.fillna(fill_value).merge(shape_column,left_index=True, right_index=True, how='inner')
    else:
        raise Exception("Dataframe does not include 'SHAPE' column")

In [4]:
if not os.path.exists('Outputs'):
    os.makedirs('Outputs')
    
outputs = ['.\\Outputs', "scratch.gdb", 'BIG5_Median_HT_Index.gdb']
gdb = os.path.join(outputs[0], outputs[1])
gdb2 = os.path.join(outputs[0], outputs[2])

if not arcpy.Exists(gdb):
    arcpy.CreateFileGDB_management(outputs[0], outputs[1])

if not arcpy.Exists(gdb2):
    arcpy.CreateFileGDB_management(outputs[0], outputs[2])

In [5]:
ht_2016 = r".\Inputs\htaindex2016_data_blkgrps_49.csv"
ht_2019 = r".\Inputs\htaindex2019_data_blkgrps_49.csv"
ht_2022 = r".\Inputs\htaindex2022_data_blkgrps_49.csv"
ht_list = [ht_2016, ht_2019, ht_2022]

block_groups = pd.DataFrame.spatial.from_featureclass(r".\Inputs\census_blockGroups_2019.shp")
block_groups2 = pd.DataFrame.spatial.from_featureclass(r".\Inputs\block_group_2024_utm12N.shp")
geog = r'E:\Tasks\WFRC_Dashboard\Inputs\Boundaries\TAZ_GeographyLookup_082025.gdb\TAZ_GeographyLookup_082025'
# city_area = pd.DataFrame.spatial.from_featureclass(r'.\Inputs\city_area.shp')
taz = r".\Inputs\WFv910_TAZ_MAG_Update.shp"

c:\Users\jreynolds\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\Lib\site-packages\pandas\core\dtypes\cast.py:1066: RuntimeWarning: invalid value encountered in cast
  if (arr.astype(int) == arr).all():
c:\Users\jreynolds\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\Lib\site-packages\pandas\core\dtypes\cast.py:1091: RuntimeWarning: invalid value encountered in cast
  if (arr.astype(int) == arr).all():
c:\Users\jreynolds\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\Lib\site-packages\pandas\core\dtypes\cast.py:1066: RuntimeWarning: invalid value encountered in cast
  if (arr.astype(int) == arr).all():
c:\Users\jreynolds\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\Lib\site-packages\pandas\core\dtypes\cast.py:1091: RuntimeWarning: invalid value encountered in cast
  if (arr.astype(int) == arr).all():


In [6]:
# # dissolve taz to city areas
# city_area = arcpy.analysis.PairwiseDissolve(
#     in_features=taz,
#     out_feature_class=os.path.join(gdb, "city_areas"),
#     dissolve_field="CITY_NAME",
#     statistics_fields=None,
#     multi_part="MULTI_PART",
#     concatenation_separator=""
# )

geog_df = pd.DataFrame.spatial.from_featureclass(geog)[['GEONAME', 'SHAPE']]
geog_df

,GEONAME,SHAPE
0,Box Elder County,"{""rings"": [[[248075.3799999999, 4653571.810000..."
1,Davis County,"{""rings"": [[[416536.51999999955, 4556181.18009..."
2,Morgan County,"{""rings"": [[[467902.5999999996, 4580340.5], [4..."
3,Salt Lake County,"{""rings"": [[[418389.3300000001, 4524991.92], [..."
4,Summit County,"{""rings"": [[[496081.8300000001, 4561991.5], [4..."
...,...,...
123,Weber County - Northern,"{""rings"": [[[417555, 4578132.800000001], [4177..."
124,Weber County - Southern,"{""rings"": [[[415626.0999999996, 4570991.9], [4..."
125,MAG,"{""rings"": [[[430973.2988, 4482106.0002], [4309..."
126,WFRC,"{""rings"": [[[411400.89969999995, 4605681.4003]..."


In [7]:
# function for processing h + t csv table
def process_ht(_ht):

    #  get year
    year = int(_ht[17:21])
    
    ht_df = pd.read_csv(_ht)
    ht_df['blkgrp'] = ht_df['blkgrp'].str.replace('"', '')

    # set zeros to NA, to avoid them being averaged
    ht_df.loc[ht_df['t_ami'] == 0,  't_ami'] = np.nan
    ht_df.loc[ht_df['h_ami'] == 0,  'h_ami'] = np.nan
    ht_df.loc[ht_df['ht_ami'] == 0,  'ht_ami'] = np.nan

    # merge
    if year <= 2019:
        ht_shp = ht_df.merge(block_groups, left_on='blkgrp', right_on='GEOID', how='left')
    else:
        ht_shp = ht_df.merge(block_groups2, left_on='blkgrp', right_on='GEOID', how='left')
    
    ht_shp = ht_shp[['blkgrp', 'h_ami', 't_ami', 'ht_ami', 't_cost_ami', 'h_cost', 'SHAPE']].copy()

    # export
    out_shp = os.path.join(gdb, f"ht_{year}")
    ht_shp.spatial.to_featureclass(location=out_shp,sanitize_columns=False) 
    return out_shp

In [8]:
geog_df2 = geog_df.copy()

for ht in ht_list:

    year = int(ht[17:21])
    ht_shp = process_ht(ht)

    # use spatial join to summarize attributes
    target_features = geog
    join_features = ht_shp
    output_features = os.path.join(gdb, "city_area_ht_sj")

    fieldmappings = arcpy.FieldMappings()
    fieldmappings.addTable(target_features)
    fieldmappings.addTable(join_features)

    # H
    fieldindex = fieldmappings.findFieldMapIndex('h_ami')
    fieldmap = fieldmappings.getFieldMap(fieldindex)
    fieldmap.mergeRule = 'median'
    fieldmappings.replaceFieldMap(fieldindex, fieldmap)

    # T
    fieldindex = fieldmappings.findFieldMapIndex('t_ami')
    fieldmap = fieldmappings.getFieldMap(fieldindex)
    fieldmap.mergeRule = 'median'
    fieldmappings.replaceFieldMap(fieldindex, fieldmap)

    # HT
    fieldindex = fieldmappings.findFieldMapIndex('ht_ami')
    fieldmap = fieldmappings.getFieldMap(fieldindex)
    fieldmap.mergeRule = 'median'
    fieldmappings.replaceFieldMap(fieldindex, fieldmap)

    # RAW T cost
    fieldindex = fieldmappings.findFieldMapIndex('t_cost_ami')
    fieldmap = fieldmappings.getFieldMap(fieldindex)
    fieldmap.mergeRule = 'median'
    fieldmappings.replaceFieldMap(fieldindex, fieldmap)

    # RAW H cost
    fieldindex = fieldmappings.findFieldMapIndex('h_cost')
    fieldmap = fieldmappings.getFieldMap(fieldindex)
    fieldmap.mergeRule = 'median'
    fieldmappings.replaceFieldMap(fieldindex, fieldmap)


    # run the spatial join
    sj = arcpy.SpatialJoin_analysis(target_features, join_features, output_features,'JOIN_ONE_TO_ONE', "KEEP_ALL", 
                            fieldmappings, "INTERSECT")

    # import into spatial dataframe
    sj_df = pd.DataFrame.spatial.from_featureclass(sj[0])[['GEONAME','h_ami', 't_ami', 'ht_ami', 't_cost_ami', 'h_cost']]
    sj_df.columns = ['GEONAME',f'h_ami_{year}', f't_ami_{year}', f'ht_ami_{year}', f't_cost_annual_{year}', f'h_cost_monthly_{year}']
    sj_df[f't_cost_monthly_{year}'] = round(sj_df[f't_cost_annual_{year}'] / 12)
    sj_df.columns

    # remaining income after expenditures
    sj_df[f'h_remain_{year}'] = 100 - sj_df[f'h_ami_{year}']
    sj_df[f't_remain_{year}'] = 100 - sj_df[f't_ami_{year}']
    sj_df[f'ht_remain_{year}'] = 100 - sj_df[f'ht_ami_{year}']

    # merge to template
    geog_df2 = geog_df2.merge(sj_df, on='GEONAME', how='left')

In [9]:
# adjust for inflation, adjust costs to 2016
geog_df2.head()

geog_df2['t_cost_annual_2016_adj'] = geog_df2['t_cost_annual_2016']
geog_df2['t_cost_annual_2019_adj'] = geog_df2['t_cost_annual_2019'] * .935
geog_df2['t_cost_annual_2022_adj'] = geog_df2['t_cost_annual_2022'] * .813

geog_df2['t_cost_monthly_2016_adj'] = geog_df2['t_cost_monthly_2016']
geog_df2['t_cost_monthly_2019_adj'] = geog_df2['t_cost_monthly_2019'] * .935
geog_df2['t_cost_monthly_2022_adj'] = geog_df2['t_cost_monthly_2022'] * .813

geog_df2['h_cost_monthly_2016_adj'] = geog_df2['h_cost_monthly_2016']
geog_df2['h_cost_monthly_2019_adj'] = geog_df2['h_cost_monthly_2019'] * .935
geog_df2['h_cost_monthly_2022_adj'] = geog_df2['h_cost_monthly_2022'] * .813

geog_df2['ht_cost_monthly_2016_adj'] = geog_df2['t_cost_monthly_2016_adj'] + geog_df2['h_cost_monthly_2016_adj']
geog_df2['ht_cost_monthly_2019_adj'] = geog_df2['t_cost_monthly_2019_adj'] + geog_df2['h_cost_monthly_2019_adj']
geog_df2['ht_cost_monthly_2022_adj'] = geog_df2['t_cost_monthly_2022_adj'] + geog_df2['h_cost_monthly_2022_adj']


In [10]:


# export
geog_df2.spatial.to_featureclass(location=os.path.join(gdb2,'BIG5_Median_HT_Index'),sanitize_columns=False) 

'e:\\Projects\\Regional-Metrics-Dashboard\\housing-plus-transportation\\Outputs\\BIG5_Median_HT_Index.gdb\\BIG5_Median_HT_Index'